!pip install pylabwons

# RESOURCES

## ASSET



| 구분 | XLE (대형주 중심) | XOP (탐사/생산 중심) | WTIU (3배 레버리지) |
| :--- | :--- | :--- | :--- |
| **기초 자산** | S&P 500 에너지 섹터 기업 | S&P Oil & Gas E&P 지수 | Solactive MicroSectors US Big Oil Index |
| **가중 방식** | **시가총액 가중** (대형주 비중↑) | **동일 가중** (모든 종목 균등 비중) | **동일 가중** (상위 12개 대형주 중심) |
| **운용 방식** | 1배수 (현물 ETF) | 1배수 (현물 ETF) | **일일 수익률 3배 추종 (ETN)** |
| **주요 종목** | 엑손모빌, 쉐브론 등 대기업 | 중소형 탐사·생산(E&P) 기업 | 엑손모빌, 쉐브론 등 우량주 12개 |
| **운용 보수** | **0.09%** | 0.35% | **0.95%** |
| **주요 특징** | 안정적인 대형주 및 배당 중심 | 유가 변동에 민감한 중소형주 포함 | 단기 방향성 매매용 (변동성 매우 높음) |


In [92]:
import yfinance as yf
import pandas as pd
import pandas_datareader.data as web
from datetime import datetime


# XOP: Energy Select Sector SPDR Fund
# XLE: SPDR S&P Oil & Gas Exploration & Production ETF
xop = yf.Ticker('XOP')
xle = yf.Ticker('XLE')
lev = yf.Ticker('WTIU')

objs = {}
for ticker in [xop, xle, lev]:
    ohlcv = ticker.history(period='10y', interval='1d')[['Open', 'High', 'Low', 'Close', 'Volume']]
    ohlcv.columns = ['open', 'high', 'low', 'close', 'volume']
    objs[ticker.ticker] = ohlcv
    

asset = pd.concat(objs, axis=1)
# asset.index = asset.index.date
asset = asset.tz_localize(None) if asset.index.tz is not None else asset
# asset

## FUTURES

In [84]:
# CL=F: WTI Crude Oil Futures
# BZ=F: Brent Crude Oil Futures
# NG=F: Natural Gas Futures
cl_f = yf.Ticker('CL=F')
bz_f = yf.Ticker('BZ=F')
ng_f = yf.Ticker('NG=F')
objs = {}
for ticker in [cl_f, bz_f, ng_f]:
    ohlcv = ticker.history(period='10y', interval='1d')[['Open', 'High', 'Low', 'Close', 'Volume']]
    ohlcv.columns = ['open', 'high', 'low', 'close', 'volume']
    objs[ticker.ticker] = ohlcv
futures = pd.concat(objs, axis=1)
# futures.index = futures.index.date

## DXY

In [85]:
# DX-Y.NYB: US Dollar Index
dxy = yf.Ticker('DX-Y.NYB').history(period='10y', interval='1d')[['Open', 'High', 'Low', 'Close', 'Volume']]
dxy.columns = ['open', 'high', 'low', 'close', 'volume']
# dxy.index = dxy.index.date

## OTHERS

* TREASURY YIELD
* CRUDE STOCKS
* BRENT GLOBAL PRICE
* OIL & GAS PPI

In [69]:
# DGS10: 10-Year Treasury Constant Maturity Rate
# T10Y2Y: 10-Year Treasury Constant Maturity Minus 2-Year Treasury
# DCOILBRENTEU: Crude Oil Prices: Brent - Europe
# PCU213111213111: Producer Price Index by Industry: Drilling Oil and Gas Wells
# BAMLH0A0HYM2: ICE BofA US High Yield Index Option-Adjusted Spread
today = datetime.now()
start = datetime(today.year - 10, today.month, today.day)
others = web.DataReader(['DGS10', 'T10Y2Y', 'DCOILBRENTEU', 'PCU213111213111', 'BAMLH0A0HYM2'], 'fred', start, today)

# ANALYTICS

In [101]:
import pandas as pd
import numpy as np
import statsmodels.tsa.stattools as ts

def relation(a: pd.Series, b: pd.Series, max_lag: int = 20, window_months: int = 6) -> pd.DataFrame:
    """
    두 시계열 데이터(a, b) 간의 시차 상관계수(CCF)를 계산하여 선/후행 관계를 분석하는 함수
    
    * NOTE
    데이터 변환       : 데이터를 로그 수익률로 변환, 시계열의 안정성(Stationarity) 확보
    슬라이딩 윈도우   : window_months 만큼 rolling하여 분석, 시간에 따른 자산 간 관계 변형 포착
    CCF(교차 상관계수): ts.ccf를 이용해 A가 n일 전/후의 움직임으로 B를 얼마나 설명하는지 수치화
    우세 관계 판별    : 선행, 후행, 동행 지표 중 절댓값이 가장 큰 항목으로 해당 시계열의 두 자산의 관계 정의

    :param a: 기준 시계열 데이터 (Pandas Series)
    :param b: 비교 대상 시계열 데이터 (Pandas Series)
    :param max_lag: 분석할 최대 시차(일수)
    :param window_months: 분석 한 번에 사용할 데이터 기간 (단위: 월)

    :return: 기간별 선/후행 판별 결과가 담긴 DataFrame
    """

    # 1. 데이터 병합 및 인덱스 통일
    df = pd.concat([a, b], axis=1).dropna()
    df.columns = ['A', 'B']
    df.index = pd.to_datetime(df.index)
    unit = '개월' if df.index.diff().min().days >= 20 else '일'
    
    # 2. 누적 상관계수 계산 (Raw 데이터 기준)
    # df와 동일한 인덱스를 가집니다.
    expanding_corr = df['A'].expanding().corr(df['B'])
    
    # 3. 수익률 변환 (CCF용)
    df_ret = np.log(df / df.shift(1)).dropna()
    
    if df_ret.empty:
        return pd.DataFrame()

    results = []
    indices = []
    
    # 기준 날짜들을 df_ret(수익률) 인덱스 기준으로 설정하여 정렬 유지
    available_dates = df_ret.index
    curr_end = available_dates[-1]
    first_date = available_dates[0]
    
    while curr_end >= first_date + pd.DateOffset(months=window_months):
        curr_start = curr_end - pd.DateOffset(months=window_months)
        
        # 윈도우 슬라이싱 (인덱스 범위 기반)
        period_ret = df_ret.loc[curr_start:curr_end]
        
        if len(period_ret) > 3 * max_lag:
            ser_a = period_ret['A']
            ser_b = period_ret['B']
            
            # --- CCF 분석 ---
            ccf_a_leads_b = ts.ccf(ser_a, ser_b, adjusted=True)[:max_lag + 1]
            ccf_b_leads_a = ts.ccf(ser_b, ser_a, adjusted=True)[:max_lag + 1]
            
            sync = ccf_a_leads_b[0]
            leading_vals = ccf_a_leads_b[1:]
            leading_idx = np.argmax(np.abs(leading_vals)) + 1
            leading = leading_vals[leading_idx - 1]
            
            lagging_vals = ccf_b_leads_a[1:]
            lagging_idx = np.argmax(np.abs(lagging_vals)) + 1
            lagging = lagging_vals[lagging_idx - 1]
            
            # --- 상관계수 추출 (에러 방지를 위해 get_loc 또는 asof 사용) ---
            # curr_end 시점에 가장 가까운 과거의 누적 상관계수 값을 가져옴
            cum_corr = expanding_corr.asof(curr_end)
            
            # 윈도우 내 Raw 상관계수
            window_raw_df = df.loc[curr_start:curr_end]
            window_raw_corr = window_raw_df['A'].corr(window_raw_df['B'])
            
            result = {
                '선행': leading,
                f'선행시차({unit})': leading_idx,
                '후행': lagging,
                f'후행시차({unit})': lagging_idx,
                '동행': sync,
                '누적(Raw)': cum_corr,
                '동행(Raw)': window_raw_corr
            }
            
            # 판별 로직
            if max(abs(leading), abs(lagging), abs(sync)) < 0.25:
                result['판별'] = '상관성 낮음'
            elif abs(leading) > abs(sync) and abs(leading) > abs(lagging):
                result['판별'] = f'{leading_idx}{unit} 선행'
            elif abs(lagging) > abs(sync) and abs(lagging) > abs(leading):
                result['판별'] = f'{lagging_idx}{unit} 후행'
            else:                
                result['판별'] = '동행'
            
            results.append(result)
            indices.append(f"{curr_start.strftime('%Y/%m')}-{curr_end.strftime('%Y/%m')}") # 인덱스를 종료일자로 명확히 표기
        
        # 3개월 전으로 이동
        curr_end = curr_end - pd.DateOffset(months=3)
        # curr_end가 실제 인덱스에 존재하지 않을 수 있으므로 가장 가까운 과거 거래일로 보정
        if curr_end >= first_date:
            curr_end = available_dates[available_dates <= curr_end][-1]

    return pd.DataFrame(results, index=indices)

# DGS10: 10-Year Treasury Constant Maturity Rate
# T10Y2Y: 10-Year Treasury Constant Maturity Minus 2-Year Treasury
# DCOILBRENTEU: Crude Oil Prices: Brent - Europe
# PCU213111213111: Producer Price Index by Industry: Drilling Oil and Gas Wells
# BAMLH0A0HYM2: ICE BofA US High Yield Index Option-Adjusted Spread

# relation(dxy['close'], asset['XLE']['close'])
# relation(futures['CL=F']['close'], asset['XLE']['close'])
# relation(futures['BZ=F']['close'], asset['XLE']['close'])
# relation(others['DCOILBRENTEU'], asset['XLE']['close'])
relation(others['PCU213111213111'].dropna(), asset['XLE']['close'].resample('MS').last(), max_lag=6, window_months=24)

,선행,선행시차(개월),후행,후행시차(개월),동행,누적(Raw),동행(Raw),판별
2024/03-2026/03,0.366409,3,-0.269313,6,-0.232405,0.916186,-0.555109,3개월 선행
2023/12-2025/12,0.407672,3,-0.165893,4,-0.273581,0.939515,-0.417953,3개월 선행
2023/09-2025/09,-0.347059,6,-0.138561,4,-0.267166,0.938771,-0.459876,6개월 선행
2023/06-2025/06,0.301511,1,-0.272089,2,-0.144156,0.937462,0.087295,1개월 선행
2023/03-2025/03,-0.480969,3,-0.350236,2,-0.085925,0.934753,0.201906,3개월 선행
2022/12-2024/12,-0.431598,6,-0.383523,2,-0.053408,0.931116,0.219467,6개월 선행
2022/09-2024/09,0.249066,1,0.497490,1,-0.257722,0.926717,0.353750,1개월 후행
2022/06-2024/06,-0.600035,3,0.414035,1,-0.215922,0.920595,0.643471,3개월 선행
2022/03-2024/03,-0.544890,3,0.332092,1,-0.144860,0.912410,0.687556,3개월 선행
2021/12-2023/12,-0.496639,3,0.431806,1,0.081572,0.904126,0.864344,3개월 선행


In [ ]:
# INITIALIZE
import matplotlib.pyplot as plt
import pandas as pd
from datetime import timedelta

def plot_dual_axis(s1: pd.Series, s2: pd.Series, **kwargs):
    # 다크 모드 적용
    plt.style.use('dark_background')
    
    # 1. 데이터 전처리: 두 데이터의 인덱스를 맞춤 (Inner Join)
    df = pd.concat([s1, s2], axis=1).dropna()
    df.columns = ['s1', 's2']

    # 3. 그래프 생성
    figsize = kwargs.get('figsize', (12, 6))
    fig, ax1 = plt.subplots(figsize=figsize)

    name1 = kwargs.get('ax1_name', s1.name if s1.name else 'Axis 1')
    name2 = kwargs.get('ax2_name', s2.name if s2.name else 'Axis 2')
    color1 = kwargs.get('color1', 'tab:blue')
    color2 = kwargs.get('color2', 'tab:orange')
    title = kwargs.get('title', f'{name1} vs {name2}')

    # 왼쪽 축
    lns1 = ax1.plot(df.index, df['s1'], color=color1, label=name1)
    ax1.set_xlabel('Date')
    ax1.set_ylabel(name1, color=color1)
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.grid(True, alpha=0.1)

    # 오른쪽 축
    ax2 = ax1.twinx()
    lns2 = ax2.plot(df.index, df['s2'], color=color2, label=name2)
    ax2.set_ylabel(name2, color=color2)
    ax2.tick_params(axis='y', labelcolor=color2)

    # 5. 범례 및 마무리
    lns = lns1 + lns2
    labs = [l.get_label() for l in lns]
    ax1.legend(lns, labs, loc='upper right')

    plt.title(title)
    fig.tight_layout()
    plt.show()


In [112]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import plotly.io as pio
# VS Code 전용 렌더러로 강제 지정
pio.renderers.default = "vscode" 

def plot_dual_axis_plotly(s1: pd.Series, s2: pd.Series, **kwargs):
    # 1. 데이터 전처리 (Inner Join)
    df = pd.concat([s1, s2], axis=1)
    df.columns = ['s1', 's2']

    # 설정값 로드
    name1 = kwargs.get('ax1_name', s1.name if s1.name else 'Axis 1')
    name2 = kwargs.get('ax2_name', s2.name if s2.name else 'Axis 2')
    color1 = kwargs.get('color1', '#1f77b4')  # tab:blue
    color2 = kwargs.get('color2', '#ff7f0e')  # tab:orange
    title = kwargs.get('title', f'{name1} vs {name2}')

    # 2. Plotly Figure 생성 (Secondary Y-axis 설정)
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # 왼쪽 축 (s1)
    fig.add_trace(
        go.Scatter(x=s1.index, y=s1, name=name1, line=dict(color=color1), xhoverformat='%Y-%m-%d', hovertemplate=f'{name1}: %{{y}}<extra></extra>'),
        secondary_y=False,
    )

    # 오른쪽 축 (s2)
    fig.add_trace(
        go.Scatter(x=s2.index, y=s2, name=name2, line=dict(color=color2), xhoverformat='%Y-%m-%d', hovertemplate=f'{name2}: %{{y}}<extra></extra>'),
        secondary_y=True,
    )

    # 3. 레이아웃 업데이트 (다크 모드 및 스타일)
    fig.update_layout(
        title_text=title,
        height=700,
        template='plotly_dark',
        hovermode='x unified',  # 마우스 오버 시 두 데이터 동시에 표시
        xaxis_title='Date',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    # 축 이름 설정
    fig.update_yaxes(title_text=name1, color=color1, secondary_y=False)
    fig.update_yaxes(title_text=name2, color=color2, secondary_y=True)

    return fig


## 01. PE

In [114]:
print(f'XLE: \n  - TRAILING PE: {xle.info.get("trailingPE")}\n  - FORWARD PE: {xle.info.get("forwardPE")}')
print(f'XOP: \n  - TRAILING PE: {xop.info.get("trailingPE")}\n  - FORWARD PE: {xop.info.get("forwardPE")}')

XLE: 
  - TRAILING PE: 20.251146
  - FORWARD PE: None
XOP: 
  - TRAILING PE: 13.171857
  - FORWARD PE: -16516.0


## 02. ASSET-DXY

### 02.01 PLOT

In [113]:
# plot_dual_axis(
plot_dual_axis_plotly(
    s1=asset['XLE']['close'],
    # s2=dxy['close'],
    s2=others['PCU213111213111'].dropna(),
    ax1_name='XLE',
    ax2_name='PCU213111213111'
).show()